In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings
from sqlalchemy import create_engine

import ast
import re

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [2]:
# 1. 통합 제품 데이터 불러오기
df_pd = pd.read_csv('data/products_merged.csv', encoding='utf-8-sig')

# 2. 통합 리뷰 데이터 불러오기
df_rv = pd.read_csv('data/reviews_merged.csv', encoding='utf-8-sig')

# 데이터가 잘 들어왔는지 상위 5개 행 확인
print("--- 제품 데이터 요약 ---")
print(df_pd.head())

print("\n--- 리뷰 데이터 요약 ---")
print(df_rv.head())

--- 제품 데이터 요약 ---
   goodsNo  플랫폼 카테고리     브랜드                         상품명       정가      판매가  \
0   876246  무신사   상의  필루미네이트       [블랙]B-스테디 하프 폴라넥-아이보리  26000.0  16900.0   
1   876277  무신사   상의  필루미네이트         [블랙]B-스테디 하프 폴라넥-블랙  26000.0  19900.0   
2   876284  무신사   상의  필루미네이트        [블랙]SET B-스테디 하프 폴라넥  52000.0  29900.0   
3   994588  무신사   상의  필루미네이트  옵티멀 베이직 셔츠-화이트[린넨＆옥스포드 선택]  53000.0  22900.0   
4   994600  무신사   상의  필루미네이트  옵티멀 베이직 셔츠-네이비[린넨&옥스포드 선택]  53000.0  22900.0   

    할인율      조회수    누적판매수      리뷰수  리뷰점수  판매상태  
0  35.0    463.0  16692.0   3685.0  94.0  SALE  
1  23.0    680.0  47033.0   9555.0  96.0  SALE  
2  43.0    574.0  67968.0  14299.0  94.0  SALE  
3  57.0  21270.0  97419.0  14475.0  96.0  SALE  
4  57.0  10423.0  14619.0   2495.0  96.0  SALE  

--- 리뷰 데이터 요약 ---
      리뷰번호  goodsNo   리뷰타입    작성자  \
0  2848952   876284  goods   tejj   
1  2850077   876284  style     '-   
2  2850085   876284  photo     '-   
3  2850098   876284  goods     '-   
4  2850905   87

In [3]:
df_products = df_pd.copy()
df_reviews = df_rv.copy()

---

# 구매옵션

In [4]:
size_pattern = re.compile(
    r'\b(XS|S|M|L|XL|2XL|3XL|SMALL|MEDIUM|LARGE|EXTRA\s*LARGE|MEDUIM|MEIDUM'
    r'|[2-9]?XL|[1-9][0-9]?(?=\b)|2[0-9]|3[0-9])\b',
    re.IGNORECASE
)

size_pattern2 = re.compile(r'(3XL|2XL|XL|XS|[SML])$')

def extract_size(val):
    if pd.isna(val):
        return None, None
    if val == 'F':
        return 'F', None
    match = size_pattern.search(val)
    if match:
        size = match.group().strip()
        option = size_pattern.sub('', val)
        option = re.sub(r'[\s·/]+', ' ', option).strip(' ·/-')
        return size, option if option else None
    return None, val

def extract_size2(val):
    if pd.isna(val):
        return None, None
    match = size_pattern2.search(val)
    if match:
        size = match.group()
        option = val[:match.start()].strip()
        return size, option if option else None
    return None, val

def extract_size_final(val):
    if pd.isna(val):
        return None, None
    result = extract_size(val)
    if result[0] is not None:
        return result
    return extract_size2(val)

# 원본 구매옵션 보존, 새 컬럼으로 분리
df_reviews[['구매사이즈', '구매상세']] = df_reviews['구매옵션'].apply(lambda x: pd.Series(extract_size_final(x)))

# 오탈자 수정
size_map = {'MEDUIM': 'MEDIUM', 'MEIDUM': 'MEDIUM'}
df_reviews['구매사이즈'] = df_reviews['구매사이즈'].replace(size_map)


---

# 키 / 몸무게

In [5]:
for col in ['키', '몸무게']:
    d = df_reviews[col]
    d_valid = d[(d >= 120) & (d <= 210)] if col == '키' else d[(d >= 30) & (d <= 200)]
    zero = (d == 0).sum()
    if col == '키':
        invalid = ((d != 0) & ((d < 120) | (d > 210))).sum()
    else:
        invalid = ((d != 0) & ((d < 30) | (d > 200))).sum()

# 이상치 및 0값을 NaN으로 변환
df_reviews['키_clean'] = df_reviews['키'].where((df_reviews['키'] >= 120) & (df_reviews['키'] <= 210), other=pd.NA)
df_reviews['몸무게_clean'] = df_reviews['몸무게'].where((df_reviews['몸무게'] >= 30) & (df_reviews['몸무게'] <= 200), other=pd.NA)

# goodsNo + 구매사이즈 그룹별 중앙값 계산 (유효값만 사용)
group_median = df_reviews.groupby(['goodsNo', '구매사이즈'])[['키_clean', '몸무게_clean']].median()

# 중앙값으로 대체
def fill_with_group_median(row, col):
    if pd.isna(row[col]):
        try:
            return group_median.loc[(row['goodsNo'], row['구매사이즈']), col]
        except KeyError:
            return row[col]
    return row[col]

df_reviews['키_clean'] = df_reviews.apply(lambda row: fill_with_group_median(row, '키_clean'), axis=1)
df_reviews['몸무게_clean'] = df_reviews.apply(lambda row: fill_with_group_median(row, '몸무게_clean'), axis=1)

# goodsNo 단독 중앙값 계산
goodsNo_median = df_reviews.groupby('goodsNo')[['키_clean', '몸무게_clean']].median()

def fill_with_goods_median(row, col):
    if pd.isna(row[col]):
        try:
            return goodsNo_median.loc[row['goodsNo'], col]
        except KeyError:
            return row[col]
    return row[col]

df_reviews['키_clean'] = df_reviews.apply(lambda row: fill_with_goods_median(row, '키_clean'), axis=1)
df_reviews['몸무게_clean'] = df_reviews.apply(lambda row: fill_with_goods_median(row, '몸무게_clean'), axis=1)

df_reviews['키'] = df_reviews['키_clean']
df_reviews['몸무게'] = df_reviews['몸무게_clean']
df_reviews.drop(columns=['키_clean', '몸무게_clean'], inplace=True)

for col in ['키', '몸무게']:
    d = df_reviews[col]
    print(f"[{col}]")
    print(f"  결측치: {d.isna().sum()}개")
    print(f"  min: {d.min()} / max: {d.max()}")
    print(f"  평균: {d.mean():.1f} / 중앙값: {d.median()}")
    if col == '키':
        outlier = d[(d < 120) | (d > 210)]
    else:
        outlier = d[(d < 30) | (d > 200)]
    print(f"  범위 밖 이상치: {len(outlier)}개")
    print()

[키]
  결측치: 336개
  min: 120.0 / max: 210.0
  평균: 172.9 / 중앙값: 174.0
  범위 밖 이상치: 0개

[몸무게]
  결측치: 338개
  min: 30.0 / max: 200.0
  평균: 68.6 / 중앙값: 68.0
  범위 밖 이상치: 0개



---

# 성별

In [6]:
df_reviews['성별'] = df_reviews['성별'].fillna('missing')

---

# 작성일

In [7]:
df_reviews['작성일'] = pd.to_datetime(df_reviews['작성일'])
# 한국 시간(Asia/Seoul)으로 변환
df_reviews['작성일'] = df_reviews['작성일'].dt.tz_convert('Asia/Seoul')
# 변환 후 시간대(+09:00) 정보 제거
df_reviews['작성일'] = df_reviews['작성일'].dt.tz_localize(None)

# 작성일 파생변수 추가
df_reviews['연도'] = df_reviews['작성일'].dt.year
df_reviews['월'] = df_reviews['작성일'].dt.month
df_reviews['일'] = df_reviews['작성일'].dt.day
df_reviews['요일'] = df_reviews['작성일'].dt.dayofweek  # 0: 월요일, 1: 화요일, ..., 6: 일요일

day_mapping = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}
df_reviews['요일'] = df_reviews['요일'].map(day_mapping)

---

# 체험단 / 사진유무

In [8]:
# str -> boolean
for col in ['체험단', '사진유무']:
    df_reviews[col] = (
        df_reviews[col]
        .astype(str)
        .str.strip()
        .str.upper()
        .map({'TRUE': True, 'FALSE': False})
        .astype('boolean')
    )

---

# 리뷰별점

In [9]:
# 0점은 무효리뷰 (2024.08 ~ 2024.10 집중발생)
# 원본 보존용 평점_raw 컬럼생성
df_reviews['평점_raw'] = df_reviews['평점']
df_reviews['평점'] = df_reviews['평점'].replace(0, np.nan)

---

# 만족도

In [10]:
#만족도: 텍스트 파싱 → 개별 컬럼 분리 + 순서형 범주형 변환
def _parse_satisfaction(text):
    if pd.isna(text) or text == "":
        return None
    result = {}
    for part in str(text).split('/'):
        if ':' in part:
            key, value = part.split(':', 1)
            result[key.strip()] = value.strip()
    return result

parsed = df_reviews['만족도'].apply(_parse_satisfaction)
sat_df = pd.DataFrame(parsed.dropna().tolist())
sat_df.index = df_reviews[df_reviews['만족도'].notnull()].index
sat_df = sat_df.reindex(df_reviews.index)

# 응답여부 플래그
df_reviews['만족도_응답여부'] = np.where(df_reviews['만족도'].notnull(), '응답', '미응답')

# 개별 항목 컬럼 추가
for col in sat_df.columns:
    df_reviews[col] = sat_df[col]

# 순서형 범주형(Ordered Categorical) 변환
categories_dict = {
    '사이즈':       ['매우 작음', '조금 작음', '정사이즈', '조금 큼', '많이 큼'],
    '화면 대비 색감': ['매우 어두움', '어두움', '화면과 비슷', '밝음', '매우 밝음'],
    '퀄리티':       ['매우 나쁨', '나쁨', '보통', '좋음', '매우 좋음'],
    '구김':         ['매우 많음', '많음', '약간 있음', '거의 없음', '전혀 없음'],
    '두께감':       ['매우 얇음', '얇음', '적당함', '두꺼움', '매우 두꺼움'],
    '신축성':       ['전혀 없음', '거의 없음', '적당함', '강함', '매우 강함'],
    '색감':         ['매우 어두움', '어두움', '화면과 비슷', '밝음', '매우 밝음'], 
    '보온성':       ['전혀 없음', '거의 없음', '적당함', '좋음', '매우 좋음'],
}


for col, order in categories_dict.items():
    if col in df_reviews.columns:
        df_reviews[col] = pd.Categorical(df_reviews[col], categories=order, ordered=True)

# 척도 유형 분류 및 수치 변환
# 유형 1 - 선형 만족도 (높을수록 좋음)
#   퀄리티, 보온성, 신축성, 두께감, 구김(반전됨)
#   → _점수 컬럼 생성 (1~5점)
# 유형 2 - 중심 기준 편차 (중간값이 이상적)
#   사이즈, 화면 대비 색감, 색감
#   → _편차 (방향 있음), _편차절대 (방향 없음, 0에 가까울수록 이상적)

# 유형 1: 선형 만족도
linear_cols = ['퀄리티', '보온성', '신축성', '두께감', '구김']

for col in linear_cols:
    if col in df_reviews.columns:
        df_reviews[f'{col}_점수'] = df_reviews[col].cat.codes.replace(-1, np.nan) + 1

print("\n 4-1. 선형 점수 변환 완료 (1~5점)")
for col in linear_cols:
    if f'{col}_점수' in df_reviews.columns:
        print(f"   {col}_점수: {df_reviews[f'{col}_점수'].value_counts(dropna=True).to_dict()}")

# 유형 2: 중심 기준 편차
center_cols = ['사이즈', '화면 대비 색감', '색감']

for col in center_cols:
    if col in df_reviews.columns:
        n      = len(df_reviews[col].cat.categories)
        center = (n - 1) / 2
        code   = df_reviews[col].cat.codes.replace(-1, np.nan)
        df_reviews[f'{col}_편차']     = code - center
        df_reviews[f'{col}_편차절대'] = (code - center).abs()


 4-1. 선형 점수 변환 완료 (1~5점)
   퀄리티_점수: {3.0: 10228, 4.0: 4019, 5.0: 3582, 2.0: 501, 1.0: 213}
   보온성_점수: {3.0: 723, 4.0: 499, 5.0: 315, 2.0: 67, 1.0: 23}
   신축성_점수: {3.0: 6832, 2.0: 863, 4.0: 786, 1.0: 433, 5.0: 249}
   두께감_점수: {3.0: 9076, 4.0: 1682, 2.0: 1505, 1.0: 244, 5.0: 211}
   구김_점수: {3.0: 1016, 4.0: 410, 2.0: 131, 5.0: 102, 1.0: 22}


---

# 중복케이스
- 완전 중복 케이스 : 작성일자를 포함한 모든 컬럼이 100% 동일한 케이스(시스템 오류)
- 중복 케이스 : 작성일자가 하루 이내이며 리뷰번호를 제외한 모든 컬럼이 동일한 케이스
- 준중복 케이스 : 작성일자가 하루 이내이며 리뷰번호 및 리뷰 내용을 제외한 모든 컬럼이 동일한 케이스
- 재구매 케이스 : 작성일자가 하루 이상 차이가 나는 케이스 / 구매 옵션이 다른 케이스 등

In [11]:
# 완전 중복 케이스
is_exact_dup_all = df_reviews.duplicated(keep=False) 
duplicate_rows_exact = df_reviews[is_exact_dup_all]

df_reviews = df_reviews.drop_duplicates().reset_index(drop=True)

# 중복 케이스
content_cols = [col for col in df_reviews.columns if col not in ['리뷰번호', '작성일']]
df_sorted = df_reviews.sort_values(by=content_cols + ['작성일']).reset_index(drop=True) 

time_diff = df_sorted.groupby(content_cols)['작성일'].diff() 
is_duplicate = time_diff <= pd.Timedelta(days=1)

duplicate_rows = df_sorted[is_duplicate]

df_reviews = df_sorted[~is_duplicate].reset_index(drop=True)

# # 준중복 케이스
# near_dup_cols = [col for col in df_reviews.columns if col not in ['리뷰번호', '리뷰내용', '작성일']]
# df_sorted_near = df_reviews.sort_values(by=near_dup_cols + ['작성일']).reset_index(drop=True)

# time_diff_prev = df_sorted_near.groupby(near_dup_cols)['작성일'].diff()
# time_diff_next = df_sorted_near.groupby(near_dup_cols)['작성일'].diff(-1).abs()

# is_near_dup = (time_diff_prev <= pd.Timedelta(days=1)) | (time_diff_next <= pd.Timedelta(days=1))

# near_dup_rows = df_sorted_near[is_near_dup]

# print("\n" + "="*60 + "\n")
# print("[준중복 케이스 분석 결과]")
# print(f"- 발견된 준중복 케이스: 총 {len(near_dup_rows)}건\n")

# if len(near_dup_rows) > 0:
#     print("[준중복 케이스 예시 확인 (상위 10건)]")
#     display(near_dup_rows.head(10))
# else:
#     print("준중복 케이스 없음")

In [19]:
# 중복으로 판정된 데이터 상위 5개 확인
print(duplicate_rows.head())

# 전체 몇 건이 중복인지 확인
print(len(duplicate_rows))

Empty DataFrame
Columns: [리뷰번호, goodsNo, 리뷰타입, 작성자, 리뷰내용, 평점, 체험단, 구매옵션, 키, 몸무게, 성별, 작성일, 만족도, 사진유무, 도움돼요, 구매사이즈, 구매상세, 연도, 월, 일, 요일, 평점_raw, 만족도_응답여부, 사이즈, 화면 대비 색감, 퀄리티, 구김, 두께감, 신축성, 색감, 보온성, 퀄리티_점수, 보온성_점수, 신축성_점수, 두께감_점수, 구김_점수, 사이즈_편차, 사이즈_편차절대, 화면 대비 색감_편차, 화면 대비 색감_편차절대, 색감_편차, 색감_편차절대]
Index: []

[0 rows x 42 columns]
0


---

# Products.csv

In [12]:
# 누적판매수 flag 생성 (누적판매수 = 0, 리뷰 수 존재)

# 누적 판매수 clean 생성 (누적판매수=0 -> NaN으로 변환)
df_products['sales_count_suspect'] = (df_products['누적판매수']==0)&(df_products['리뷰수']>0)
df_products['sales_count_clean'] = np.where(df_products['sales_count_suspect'],np.nan, df_products['누적판매수'])

# 조회수 flag 생성 (조회수 = 0 or 조회수 < 누적판매수)
df_products['view_count_suspect'] = (df_products['조회수'] == 0) | (df_products['조회수'] < df_products['누적판매수'])


#수정 부분
# 저판매 상품 컬럼 생성 (리뷰수가 0개 혹은 누적판매수가 50개 이하인 상품 )
# 누적판매수 < 50개 이하인 상품은 누적판매수가 =0 으로 찍히는 패턴을 파악
# 누적판매수=0이지만 리뷰수가 존재하는 상품 -> 리뷰개수 확인결과 평균 2~3개, 최대 16개로 판매가 저조한 상품임을 확인
df_products['inactive_candidate'] = (df_products['리뷰수'] == 0) | (df_products['누적판매수']<50)

In [13]:
df_products['sales_count_suspect'].value_counts()

sales_count_suspect
False    2245
True      741
Name: count, dtype: int64

In [14]:
#1. 리뷰 텍스트 / 키워드 분석용 Subset
df_review_text = df_products[df_products['리뷰수']>0].copy()
df_review_text = df_review_text[['goodsNo','플랫폼','카테고리','브랜드','상품명','정가','판매가','할인율','조회수','누적판매수','리뷰수','리뷰점수']]

#2. 판매 성과 / 전환율 분석용 Subset
df_performance = df_products[(df_products['sales_count_suspect']== False) & (df_products['view_count_suspect']== False)].copy()
df_performance = df_performance[['goodsNo','플랫폼','카테고리','브랜드','상품명','정가','판매가','할인율','조회수','누적판매수','리뷰수','리뷰점수']]

#3. 저판매 상품 관리용 Subset
df_inactive = df_products[df_products['inactive_candidate']==True].copy()
df_inactive = df_inactive[['goodsNo','플랫폼','카테고리','브랜드','상품명','정가','판매가','할인율','조회수','누적판매수','리뷰수','리뷰점수']]

### CSV파일로 저장

In [15]:
df_reviews = df_reviews.rename(columns={
    "화면 대비 색감": "화면대비색감",
    "화면 대비 색감_편차": "화면대비색감_편차",
    "화면 대비 색감_편차절대": "화면대비색감_편차절대"
})

In [16]:
df_reviews.to_csv("data/cleaned_reviews.csv", index=False, encoding='utf-8-sig')
df_products.to_csv("data/cleaned_products.csv", index=False, encoding='utf-8-sig')

In [17]:
cr = pd.read_csv("data/cleaned_reviews.csv")
cp = pd.read_csv("data/cleaned_products.csv")

cr.shape, cp.shape

((688713, 42), (2986, 17))

((685292, 45), (2948, 17))


In [18]:
cr['작성일'].max()

'2026-04-29 23:49:07'